In [22]:
import torchvision
import torch
import numpy as np
import math
from PIL import Image
import glob
import os
import cv2
import matplotlib.pyplot as plt

np.set_printoptions(threshold = np.inf)

The following part contains kernel, find_median, median, bound and unbound functions. They are side functions that helps us to perform other functions. Kernel groups our image with given size. median blurs a little bit the image. bound and unbound gives dummy pixels to our image. Im not currently using some of these functions in my code because I do some tryings and I consider them.

In [23]:
def kernel(image, x,y, size=1):
    return image[:,x-size:x+size+1, y-size:y+size+1]

def find_median(kernel):
    arr = kernel.flatten()
    return np.median(arr)

def median(image, size=1):
    shape = image.shape
    image = bound(image, size)
    new_image = np.zeros((image.shape))
    for i in range(size,shape[1]+size):
        for j in range(size,shape[2]+size):
            ij_kernel = kernel(image,i,j,size)
            for k in range(shape[0]):
                new_image[k,i,j] = find_median(ij_kernel[k])
    new_image = unbound(new_image, size)
    return new_image.astype(np.uint8)

def bound(image, size=1):
    for _ in range(size):
        image = np.insert(image, 0, 0, axis=1)
        image = np.insert(image, image.shape[1], 0, axis=1)
        image = np.insert(image, 0, 0, axis=2)
        image = np.insert(image, image.shape[2], 0, axis=2)
        for j in range(image.shape[1]):
            image[:,j,0] = image[:,j,1]
            image[:,j,image.shape[2]-1] = image[:,j,image.shape[2]-2]
        for j in range(image.shape[2]):
            image[:,0,j] = image[:,1,j]
            image[:,image.shape[1]-1,j] = image[:,image.shape[1]-2,j]

    return image

def unbound(image, size=1):
    for _ in range(size):
        image = np.delete(image, 0, axis=1)
        image = np.delete(image, 0, axis=2)
        image = np.delete(image, image.shape[1]-1, axis=1)
        image = np.delete(image, image.shape[2]-1, axis=2)
    return image

Next part got the sobel algorithm. We have 3 options according to axis parameter (xy, x, y).

In [24]:
def sobel(image, axis="xy",size=1):
    Gx = np.array([[-1,0,1],[-2,0,2],[-1,0,1]])
    Gy = np.array([[-1,-2,-1],[0,0,0],[1,2,1]])
    shape = image.shape
    image = bound(image, size)
    image_copy = np.zeros((image.shape))
    for i in range(size,shape[1]+size):
        for j in range(size,shape[2]+size):
            ij_kernel = kernel(image,i,j,size)
            for k in range(shape[0]):
                #image_copy[k,i,j] = np.sqrt(np.square(np.sum(Gx*ij_kernel[k])) + np.square(np.sum(Gy*ij_kernel[k])))
                if(axis=="x"):
                    image_copy[k,i,j] = abs(np.sum(Gx*ij_kernel[k]))
                elif(axis=="y"):
                    image_copy[k,i,j] = abs(np.sum(Gy*ij_kernel[k]))
                else:
                    image_copy[k,i,j] = abs(np.sum(Gx*ij_kernel[k])) + abs(np.sum(Gy*ij_kernel[k]))
    image_copy *= 255.0 / image_copy.max()
    image_copy = unbound(image_copy, size)
    return image_copy.astype(np.uint8)



My canny algorithm got threshold at 70. I tried with 40 too but in my opinion, the results are better at 70.

In [25]:
def canny(image,size=1):
    Gx = np.array([[-1,0,1],[-2,0,2],[-1,0,1]])
    Gy = np.array([[-1,-2,-1],[0,0,0],[1,2,1]])
    shape = image.shape
    image = bound(image, size)
    image_copy_x = np.zeros(image.shape)
    image_copy_y = np.zeros(image.shape)

    for i in range(size,shape[1]+size):
        for j in range(size,shape[2]+size):
            ij_kernel = kernel(image,i,j,size)
            for k in range(shape[0]):
                image_copy_x[k,i,j] = np.sum(Gx*ij_kernel[k])
                image_copy_y[k,i,j] = np.sum(Gy*ij_kernel[k])
                # image_copy_x[k,i,j] = abs(np.sum(Gx*image[k,i-size:i+size+1, j-size:j+size+1]))
                # image_copy_y[k,i,j] = abs(np.sum(Gy*image[k,i-size:i+size+1, j-size:j+size+1]))
    max_copy = max(image_copy_x.max(), image_copy_y.max())
    image_copy_x *= 255.0 / max_copy
    image_copy_y *= 255.0 / max_copy
    directions = np.arctan2(image_copy_y, image_copy_x)
    directions[directions < 0] += np.pi
    magnitudes = np.sqrt(np.square(image_copy_x) + np.square(image_copy_y))
    return_magnitudes = magnitudes.copy()
    for i in range(size, shape[1] + size):
        for j in range(size, shape[2] + size):
            for k in range(shape[0]):
                if 0 <= directions[k, i, j] < np.pi / 8 or 7 * np.pi / 8 <= directions[k, i, j] <= np.pi:
                    neighbors = [magnitudes[k, i, j-1], magnitudes[k, i, j+1]]
                elif np.pi / 8 <= directions[k, i, j] < 3 * np.pi / 8:
                    neighbors = [magnitudes[k, i-1, j-1], magnitudes[k, i+1, j+1]]
                elif 3 * np.pi / 8 <= directions[k, i, j] < 5 * np.pi / 8:
                    neighbors = [magnitudes[k, i-1, j], magnitudes[k, i+1, j]]
                else:
                    neighbors = [magnitudes[k, i-1, j+1], magnitudes[k, i+1, j-1]]
                if magnitudes[k, i, j] <= max(neighbors):
                    return_magnitudes[k, i, j] = 0


    return_magnitudes *= 255.0 / return_magnitudes.max()
    return_magnitudes = unbound(return_magnitudes, size)
    return_magnitudes = np.where(return_magnitudes > 70, 255, return_magnitudes)
    return_magnitudes = np.where(return_magnitudes < 70, 0, return_magnitudes)

    return return_magnitudes.astype(np.uint8)


In the next part, I tried lots of filter. Actually, I could not handle it effectively so I'm not using them at my last version.

In [26]:
def white_finder(image):
    shape = image.shape
    average = np.mean(image)
    for i in range(shape[0]):
        for j in range(shape[1]):
            for k in range(shape[2]):
                if image[i,j,k] > average:
                    image[i,j,k] = 255
                else:
                    image[i,j,k] = 0
    return image

def dark_killer(image):
    shape = image.shape
    for i in range(shape[1]):
        for j in range(shape[2]):
            if image[0,i,j] < 100 and image[1,i,j] < 100 and image[2,i,j] < 100:
                    image[:,i,j] = 0
    return image

def colour_killer(image):
    shape = image.shape
    for i in range(shape[1]):
        for j in range(shape[2]):
            if abs(int(image[0,i,j]) - int(image[1,i,j])) > 40 or abs(int(image[0,i,j]) - int(image[2,i,j])) > 40 or abs(int(image[1,i,j]) - int(image[2,i,j])) > 40:
                image[0,i,j] = 100
                image[1,i,j] = 100
                image[2,i,j] = 100
    return image
def non_linear_lower_contrast(image):
    shape = image.shape
    for i in range(shape[0]):
        for j in range(shape[1]):
            for k in range(shape[2]):
                image[i,j,k] = 255 * ((image[i,j,k]/255) ** 1/3)
    return image
def vertical_votes_finder(image,votes, thetas, ps):
    vote_shape = votes.shape
    vertical_votes = np.zeros((vote_shape[0],vote_shape[1]))
    for i in range(vote_shape[0]):
        for j in range(vote_shape[1]):
            if votes[i,j] == 0:
                continue
            theta = thetas[j]
            if(theta < np.pi/8 and theta >= -np.pi/8):
                vertical_votes[i,j] = votes[i,j]
                
    return vertical_votes

def horizontal_votes_finder(image,votes, thetas, ps):
    vote_shape = votes.shape
    horizontal_votes = np.zeros((vote_shape[0],vote_shape[1]))
    for i in range(vote_shape[0]):
        for j in range(vote_shape[1]):
            if votes[i,j] == 0:
                continue
            theta = thetas[j]
            if((theta > 3*np.pi/8 and theta <= 5*np.pi/8) or (theta < -3*np.pi/8 and theta >= -5*np.pi/8)):
                horizontal_votes[i,j] = votes[i,j]
                
    return horizontal_votes

Here I have lots of functions. I have 2 hough algorithms. One of them is to find lines. Other one is generalised hough algorithm to find some points which are possible license plate areas. And I'm combining these two hough algorithms to choose best lines.

hough: It is not the generalised one. This one is for the lines. At the last part of hough, I'm grouping the votes. Every 3x3 block accumulates to the their biggest index. So we earn some time with this.

These functions are in the grave (I'm not using them, I tried and they didn't work great):
print_hough_lines, rectangle_finder, draw_rectangles, resize_image, rotate_image, parallel_lines_horizontal

hough_lines_max: It takes votes, creates lines with them, splits the biggest lines to shorter lines that we may need. It has lots of parameters. vertical is 1 if we want vertical lines and 0 if we want horizontals. Count represents how many lines we want. tan and tan_2 represents our lines m limits. This function eliminates similiar lines at the end and returns a line array.

draw_lines: Draw lines.

generate_plate_template: It simply generates a plate with given sizes. I used 120x50 plate in my generalized hough algorithm.

edge_rotation, r_table, compute_accumulator_array: These functions are totally a complete generalised hough algorithm. But I have little changes. I'm using both sobel and canny in it but I don't use orientations at every part. Because I don't know the center of the plate. It may have different letters with different orientations (In pixel world). So I changed some orientation part. And It worked.

vote_visualiser: It helped me to plot vote map.

rgb2gray: It makes gray the rgb images.

accumulator_pointer: Actually It uses both lines and generalised accumulator map to find the best area to be the license plate. It searches possible areas and give scores to them with the help of the lines and the accumulator points.

find_the_plate: It has 3 parts. First one uses horizontal lines that is given from hough. If we have parallel lines that satisfies some conditions, it returns. Else: it looks for the vertical lines. Else: it calls accumulator_pointer and tries to find the plate.


In [27]:

def hough(image):
    shape = image.shape
    height = shape[1]
    width = shape[2]
    diagonal = int(np.round(np.sqrt(height**2 + width ** 2)))
    thetas = np.deg2rad(np.arange(0, 180))
    ps = np.linspace(-diagonal, diagonal, 2*diagonal)
    votes = np.zeros((2*diagonal, len(thetas)))
    for i in range(shape[0]):
        for j in range(height):
            for k in range(width):
                if image[i,j,k] == 255:
                    for a in range(len(thetas)):
                        p = int(k * math.cos(thetas[a]) + j * math.sin(thetas[a]))
                        votes[(p + diagonal),a] += 1
        
    for i in range(2,votes.shape[0],3):
        for j in range(2,votes.shape[1],3):
            max_vote = np.max(votes[i-2:i+1,j-2:j+1])
            max_vote_index = np.where(votes[i-2:i+1, j-2:j+1] == max_vote)
            total = np.sum(votes[i-2:i+1,j-2:j+1])
            votes[i-2:i+1,j-2:j+1] = 0
            votes[i-2+max_vote_index[0][0],j-2+max_vote_index[1][0]] = total
    return votes, thetas, ps

def print_hough_lines(image,votes, thetas, ps, threshold=10):
    lines = []
    height = image.shape[1]
    width = image.shape[2]
    for i in range(votes.shape[0]):
        for j in range(votes.shape[1]):
            if votes[i,j] > threshold:
                theta = thetas[j]
                p = ps[i] 
                sin = np.sin(theta)
                cos = np.cos(theta)
                if sin == 0:
                    y1 = 0
                    x1 = int((p - y1 * sin) / cos)
                    y2 = height
                    x2 = int((p - y2 * sin) / cos)
                else:
                    x1 = 0
                    y1 = int((p - x1 * cos) / sin)
                    x2 = width
                    y2 = int((p - x2 * cos) / sin)
                lines.append((x1, y1, x2, y2))
    return lines

def hough_lines_max(image,votes, thetas, ps, threshold=10,vertical=1,count=5,tan = 0.3,tan_2 = 5):
    lines = []
    height = image.shape[1]
    width = image.shape[2]
    for i in range(votes.shape[0]):
        for j in range(votes.shape[1]):
            if votes[i,j] > threshold:
                theta = thetas[j]
                p = ps[i]
                sin = np.sin(theta)
                cos = np.cos(theta)
                if(vertical == 1 and cos != 0):
                    y1 = 0
                    x1 = int((p -y1 *sin)/ cos)
                    y2 = height
                    x2 = int((p -y2 *sin)/ cos)
                    if(x2 - x1 == 0):
                        m = 999
                    else:
                        m = (y2 - y1) / (x2 - x1)
                    if(m > tan_2 or m < -tan_2):
                        gap = 0
                        streak = 0
                        x_start = x1
                        x_end = x2
                        y_start = y1
                        y_end = y2
                        for k in range(height):
                            x = int((p - k * sin) / cos)
                            if(x>width-1 or x<0):
                                break
                            elif(x == 0):
                                x=1
                            elif(x == width-1):
                                x = width-2
                            if(image[0,k,x] == 255 or image[0,k,x+1] == 255 or image[0,k,x-1] == 255):
                                y_end = k
                                x_end = x
                                streak += 1
                            else:
                                gap+=1
                                if ( gap > 3):
                                    if streak>5:
                                        lines.append((x_start, y_start, x_end, y_end, m))
                                    streak = 0
                                    gap = 0
                                    x_start = x
                                    y_start = k
                                else:
                                    streak+=1
                                    y_end = k
                                    x_end = x
                elif(vertical == 0 and sin != 0):
                    x1 = 0
                    y1 = int((p - x1 * cos) / sin)
                    x2 = width
                    y2 = int((p - x2 * cos) / sin)
                    m = (y2 - y1) / (x2 - x1)
                    if(m < tan and m > -tan):
                        gap = 0
                        streak = 0
                        x_start = x1
                        x_end = x2
                        y_start = y1
                        y_end = y2
                        for k in range(width):
                            y = int((p - k * cos) / sin)
                            if(y>height-1 or y<0):
                                break
                            elif(y == 0):
                                y=1
                            elif(y == height-1):
                                y = height-2
                            if(image[0,y,k] == 255 or image[0,y+1,k] == 255 or image[0,y-1,k] == 255):
                                x_end = k
                                y_end = y
                                streak += 1
                            else:
                                
                                gap+=1
                                if ( gap > 3):
                                    if streak>20:
                                        lines.append((x_start, y_start, x_end, y_end, m))
                                    streak = 0
                                    gap = 0
                                    x_start = k
                                    y_start = y
                                else:
                                    streak+=1
                                    x_end = k
                                    y_end = y
                        
    
    lines = sorted(lines, key=lambda x: np.sqrt((x[0]-x[2])**2 + (x[1]-x[3])**2),reverse=True)
    a = 999
    if len(lines) > count:
        counted_lines = lines[:count]
    else:
        counted_lines = lines
    while a != 0:
        deleted_index = []
        a = 0
        for i in range(len(counted_lines)):
            for j in range(i+1,len(counted_lines)):
                if abs(lines[i][0]-lines[j][0]) < 5 and abs(lines[i][1]-lines[j][1]) < 5 and abs(lines[i][2]-lines[j][2]) < 5 and abs(lines[i][3]-lines[j][3]) < 5:
                    deleted_index.append(j)
        deleted_index = sorted(set(deleted_index), reverse=True)
        for i in deleted_index:
            counted_lines.pop(i)
            a+=1
    if len(lines) > count:
        counted_lines = lines[:count]
    else:
        counted_lines = lines
    return counted_lines

def rectangle_finder(vertical_lines, horizontal_lines):
    neighbours = []
    rectangles_level_1 = []
    rectangles_level_2 = []
    rectangles_level_3 = []
    rectangles_level_4 = []
    got_rectangle_level_1 = False
    for i in horizontal_lines:
        x1, y1, x2, y2 = i
        neighbour_1 = []
        neighbour_2 = []
        for j in vertical_lines:
            x3, y3, x4, y4 = j
            if abs(x3-x1) < 5 and (abs(y4-y1) < 5 or abs(y3-y1) < 5):
                neighbour_1.append(j)
            elif abs(x4-x1) < 5 and (abs(y4-y1) < 5 or abs(y3-y1) < 5):
                neighbour_1.append(j)
            if abs(x3-x2) < 5 and (abs(y4-y2) < 5 or abs(y3-y2) < 5):
                neighbour_2.append(j)
            elif abs(x4-x2) < 5 and (abs(y4-y2) < 5 or abs(y3-y2) < 5):
                neighbour_2.append(j)
        neighbours.append(neighbour_1)
        neighbours.append(neighbour_2)
    for i in range(len(horizontal_lines)):
        has_twins = False
        for j in range(i+1,len(horizontal_lines)):
            line_i = horizontal_lines[i]
            line_j = horizontal_lines[j]
            m_i = (line_i[3] - line_i[1]) / (line_i[2] - line_i[0])
            m_j = (line_j[3] - line_j[1]) / (line_j[2] - line_j[0])
            if abs(m_i - m_j) < 0.3 and abs(line_i[1] - line_j[1]) > 10:
                neighbours_i_1 = neighbours[2*i]
                neighbours_i_2 = neighbours[2*i+1]
                neighbours_j_1 = neighbours[2*j]
                neighbours_j_2 = neighbours[2*j+1]
                same_neighbour_1 = 0
                same_neighbour_2 = 0
                for k in neighbours_i_1:
                    if k in neighbours_j_1:
                        same_neighbour_1 = k
                for k in neighbours_i_2:
                    if k in neighbours_j_2:
                        same_neighbour_2 = k
                if same_neighbour_1 and same_neighbour_2:
                    has_twins = True
                    got_rectangle_level_1 = True
                    rectangles_level_1.append((line_i, line_j, same_neighbour_1, same_neighbour_2))
        if not has_twins and not got_rectangle_level_1:
            neighbour_1 = neighbours[2*i]
            neighbour_2 = neighbours[2*i+1]
            for j in range(len(neighbour_1)):
                for k in range(len(neighbour_2)):
                    vertical_1 = neighbour_1[j]
                    vertical_2 = neighbour_2[k]
                    if(vertical_1[2] - vertical_1[0]) == 0:
                        vertical_1_m = 999
                    else:
                        vertical_1_m = (vertical_1[3] - vertical_1[1]) / (vertical_1[2] - vertical_1[0])
                    if(vertical_2[2] - vertical_2[0]) == 0:
                        vertical_2_m = 999
                    else:
                        vertical_2_m = (vertical_2[3] - vertical_2[1]) / (vertical_2[2] - vertical_2[0])
                    vertical_1_length = vertical_1[3] - vertical_1[1]
                    vertical_2_length = vertical_2[3] - vertical_2[1]
                    if abs(vertical_1_m - vertical_2_m) < 0.3 and abs(vertical_1[1] - vertical_2[1]) < 5 and abs(vertical_1_length - vertical_2_length) < 5:
                        rectangles_level_2.append((horizontal_lines[i], vertical_1, vertical_2))            
    return rectangles_level_1                   
def draw_lines(image, lines):
    return_image = np.zeros((1,image.shape[1],image.shape[2]))
    for line in lines:
        x1, y1, x2, y2, m = line
        return_image[0] = cv2.line(return_image[0], (int(x1), int(y1)), (int(x2), int(y2)), (255), 2)
    return return_image.astype(np.uint8)

def draw_rectangles(image, rectangles):
    return_image = np.zeros((1,image.shape[1],image.shape[2]))
    for rectangle in rectangles:
        line1, line2, same_neighbour_1, same_neighbour_2 = rectangle
        x1, y1, x2, y2 = line1
        x3, y3, x4, y4 = line2
        x5, y5, x6, y6 = same_neighbour_1
        x7, y7, x8, y8 = same_neighbour_2
        return_image[0] = cv2.line(return_image[0], (int(x1), int(y1)), (int(x2), int(y2)), (255), 2)
        return_image[0] = cv2.line(return_image[0], (int(x3), int(y3)), (int(x4), int(y4)), (255), 2)
        return_image[0] = cv2.line(return_image[0], (int(x5), int(y5)), (int(x6), int(y6)), (255), 2)
        return_image[0] = cv2.line(return_image[0], (int(x7), int(y7)), (int(x8), int(y8)), (255), 2)
        
    return return_image.astype(np.uint8)
def generate_plate_template(width, height):
    template = np.ones((1,height, width), dtype=np.uint8)

    return template

def resize_image(image, scale):
    h, w = image.shape[:2]
    new_h, new_w = int(h * scale), int(w * scale)
    resized_image = np.zeros((new_h, new_w), dtype=image.dtype)
    
    for i in range(new_h):
        for j in range(new_w):
            # Yeni boyutlara göre orijinal görüntüdeki piksel değerlerini kopyala
            resized_image[i, j] = image[int(i / scale), int(j / scale)]
    
    return resized_image

def rotate_image(image, angle):
    angle_rad = np.radians(angle)
    cos_theta = np.cos(angle_rad)
    sin_theta = np.sin(angle_rad)
    
    h, w = image.shape[:2]
    cx, cy = w // 2, h // 2 

    rotated_image = np.zeros_like(image)
    for i in range(h):
        for j in range(w):
            x = j - cx
            y = i - cy
            new_x = np.round(x * cos_theta - y * sin_theta + cx).astype(int)
            new_y = np.round(x * sin_theta + y * cos_theta + cy).astype(int)
            if 0 <= new_x < w and 0 <= new_y < h:
                rotated_image[new_y, new_x] = image[i, j]

    return rotated_image
#def r-table():

def edge_rotation(image):
    dx = sobel(image, "x")
    dy = sobel(image, "y")
    returner = np.arctan2(dy, dx) + np.pi
    #returner = np.where(np.logical_and(dx < 0.1, dx > -0.1), 0, returner)

    return returner

def r_table(origin, row_count, agnostic_center=True):
    edges = generate_plate_template(120, 30)
    rotations = edge_rotation(edges)
    r_table = [[] for i in range(row_count)]

    for (coord, value) in np.ndenumerate(edges):
        i = coord[1]
        j = coord[2]
        if value:
            r = np.sqrt((origin[0] - i) ** 2 + (origin[1] - j) ** 2)
            if agnostic_center and (i == 0 or j == 0 or i == edges.shape[1] - 1 or j == edges.shape[2] - 1):
                alpha = 0
            else:
                alpha = np.arctan2(i - origin[0], j - origin[1]) + np.pi

            index = int(row_count * rotations[0, i, j] / (2 * np.pi))
            r_table[index].append((r, alpha, edges[0, i, j]))
    return r_table

def compute_accumulator_array(edges,image, r_table):
    rotations = edge_rotation(image)
    accumulator = np.zeros(image.shape)
    r_table_row_count = len(r_table)
    
    for (coord, value) in np.ndenumerate(edges):
        i = coord[1]
        j = coord[2]
        if value:
            index = int(r_table_row_count * rotations[0,i, j] / (2 * np.pi))
            r_row = r_table[index]
            for (r,alpha,weight) in r_row:

                if alpha == 0:
                    accum_i = i
                    accum_j = j
                else:
                    accum_i = int(i + r * np.sin(alpha))
                    accum_j = int(j + r * np.cos(alpha))

                if accum_i < accumulator.shape[1] and accum_j < accumulator.shape[2] and accum_i > 0 and accum_j > 0:
                    accumulator[0,accum_i, accum_j] += weight
    accumulator = accumulator / accumulator.max() * 255
    return accumulator.astype(np.uint8)


def vote_visualiser(votes):
    return_image = np.zeros((1,votes.shape[0],votes.shape[1]))
    for i in range(votes.shape[0]):
        for j in range(votes.shape[1]):
            return_image[0,i,j] = votes[i,j]

    return_image = return_image.astype(np.uint8)
    torchvision.io.write_png(torch.from_numpy(return_image), 'plot.png')

def parallel_lines_horizontal(lines):
    parallel_lines = []
    for i in range(len(lines)):
        added = False
        for j in range(i+1, len(lines)):
            x1, y1, x2, y2 = lines[i]
            x3, y3, x4, y4 = lines[j]
            i_m = (y2 - y1) / (x2 - x1)
            j_m = (y4 - y3) / (x4 - x3)
            length = np.sqrt((x3 - x1) ** 2 + (y3 - y1) ** 2)
            if abs(i_m - j_m) < 0.017 and length < 5 and i_m > -0.1 and i_m < 0.1:
                if not added:
                    added = True
                    parallel_lines.append((x1, y1, x2, y2))
                parallel_lines.append((x3, y3, x4, y4))
    return parallel_lines

def rgb2gray(rgb):
    for i in range(rgb.shape[1]):
        for j in range(rgb.shape[2]):
            rgb[0,i,j] = 0.299 * rgb[0,i,j] + 0.587 * rgb[1,i,j] + 0.114 * rgb[2,i,j]
    
    return rgb[:1]

def accumulator_pointer(accumulator,x,y, verticals, horizontals):
    height = verticals.shape[1]
    width = verticals.shape[2]
    left = x
    right = x
    up = y
    down = y
    for i in range(y, height):
        if horizontals[0,i,x] == 255:
            down = i
            break

    for i in range(y, 0, -1):
        if horizontals[0,i,x] == 255:
            up = i
            break
    slope = 0
    for i in range(x, width):
        if up+slope+1 >= height:
            slope = height - up - 2
        if horizontals[0,up+slope,i] == 0:
            if horizontals[0,up+slope+1,i] == 255:
                slope += 1
            elif horizontals[0,up+slope-1,i] == 255:
                slope -= 1
            else:
                right = i
                break
    
    for i in range(x, 0, -1):
        if up+slope >= height:
            slope = height - up - 1
        if horizontals[0,up+slope,i] == 0:
            if horizontals[0,up+slope+1,i] == 255:
                slope += 1
            elif horizontals[0,up+slope-1,i] == 255:
                slope -= 1
            else:
                left = i
                break

    for i in range(1,10):
        if(down+i>=height):
            down = height-1
            break
        if(horizontals[0,down+i,x]!=255):
            down = down+i-1
            break
    for i in range(1,10):
        if(up-i<0):
            up = 0
            break
        if(horizontals[0,up-i,x]!=255):
            up = up-i+1
            break
    for i in range(1,6):
        if(right+i>=width):
            right = width-1
            break
        if y+slope >= height:
            slope = height - y - 1
        if(verticals[0,y+slope,right+i]!=255):
            right = right+i-1
            break
    for i in range(1,6):
        if(left-i<0):
            left = 0
            break
        if(verticals[0,y,left-i]!=255):
            left = left-i+1
            break
    x1 = left
    x2 = right
    y1 = up
    y2 = down
    y3 = up + slope
    y4 = down + slope
    point = abs((left-right)*(up-down))
    total = 0
    white = 0
    for i in range(up, down):
        total += 2
        if(i+slope >= height):
            slope = height - i - 1
        if(left == 0):
            left = 1
        if(right == width-1):
            right = width-2
        if verticals[0,i,left] == 255 or verticals[0,i,left+1] == 255 or verticals[0,i,left-1] == 255:
            white += 1
        if verticals[0,i+slope,right] == 255 or verticals[0,i+slope,right+1] == 255 or verticals[0,i+slope,right-1] == 255:
            white += 1
    if(left-right == 0 or up-down == 0 or abs(up-down)/abs(left-right) > 0.7 or abs(left-right)/abs(up-down) > 5):
        point = 0.005*point
    elif(white/total < 0.5):
        point = 0.1*point

    return( point,x1,x2,y1,y2,y3,y4)
    

    
def find_the_plate(accumulator, vertical_lines, horizontal_lines, verticals, horizontals):
    height = accumulator.shape[1]
    width = accumulator.shape[2]
    possible_lines = []
    for i in range(height):
        for j in range(width):
            if accumulator[0,i,j] > 100:
                top_lines = []
                bottom_lines = []
                for h in horizontal_lines:
                    if h[1] < i and h[3] < i and h[0] < j and h[2] > j:   
                        top_lines.append(h)
                    if h[1] > i and h[3] > i and h[0] < j and h[2] > j:
                        bottom_lines.append(h)
                top_lines = sorted(top_lines, key=lambda x: x[1],reverse=True)
                bottom_lines = sorted(bottom_lines, key=lambda x: x[1])
                if top_lines != [] and bottom_lines != []:
                    for top_line in top_lines:
                        for bottom_line in bottom_lines:
                            if abs(top_line[4]- bottom_line[4]) < 0.1 and abs(top_line[0] - bottom_line[0]) < 5 and abs(top_line[2] - bottom_line[2]) < 5 and 7.1 > abs(top_line[2] - top_line[0])/abs(top_line[1]-bottom_line[1]) > 1.5:
                                point = np.sum(accumulator[0,top_line[1]:bottom_line[1],top_line[0]:top_line[1]])
                                possible_lines.append((top_line, bottom_line, point))
                                break
    
    possible_lines = sorted(possible_lines, key=lambda x: x[2],reverse=True)
    if len(possible_lines) > 0:
        return possible_lines[0]
    else:
        for i in range(height):
            for j in range(width):
                if accumulator[0,i,j] > 100:
                    left_lines = []
                    right_lines = []
                    for h in vertical_lines:
                        if h[0] < j and h[2] < j and h[1] < i and h[3] > i:   
                            left_lines.append(h)
                        if h[0] > j and h[2] > j and h[1] < i and h[3] > i:
                            right_lines.append(h)
                    left_lines = sorted(left_lines, key=lambda x: x[0],reverse=True)
                    right_lines = sorted(right_lines, key=lambda x: x[0])
                    if left_lines != [] and right_lines != []:
                        for left_line in left_lines:
                            for right_line in right_lines:
                                if  abs(left_line[1] - right_line[1]) < 10 and abs(left_line[3] - right_line[3]) < 10 and 0.14 < abs(left_line[3] - left_line[1])/abs(left_line[0]-right_line[0]) < 0.65:
                                    point = np.sum(accumulator[0,left_line[1]:left_line[3],left_line[0]:right_line[0]])
                                    possible_lines.append((left_line, right_line, point))
                                    break
        possible_lines = sorted(possible_lines, key=lambda x: x[2],reverse=True)
        if len(possible_lines) > 0:
            return possible_lines[0]
        else:
            accumulator_point_table = np.zeros((accumulator.shape[1],accumulator.shape[2],7))
            for i in range(accumulator.shape[1]):
                for j in range(accumulator.shape[2]):
                    if(accumulator[0,i,j] > 200):
                        accumulator_point_table[i,j] = accumulator_pointer(accumulator,j,i,verticals,horizontals)
            accumulator_point_table_max = accumulator_point_table[:,:,0].max()
            max_value_index = np.where(accumulator_point_table[:,:,0] == accumulator_point_table_max)
            origin_x = max_value_index[1][0]
            origin_y = max_value_index[0][0]
            left = int(accumulator_point_table[origin_y,origin_x,1])
            right = int(accumulator_point_table[origin_y,origin_x,2])
            up = int(accumulator_point_table[origin_y,origin_x,3])
            down = int(accumulator_point_table[origin_y,origin_x,4])
            up_slope = int(accumulator_point_table[origin_y,origin_x,5])
            down_slope = int(accumulator_point_table[origin_y,origin_x,6])
            line_1 = [left,up,right,up_slope,1]
            line_2 = [left,down,right,down_slope,1]
            return [line_1,line_2,0]
            


I have printed out the dataset with 3 different parameter set: 120-80, 60-40, and 30-20. These numbers represents the number of horizontal and vertical lines that is used when called hough_lines_max. Actually, I have similiar results but the best one is 60-40

The results are:
30-20-> 201 False, 125 True, 107 SemiTrue.
60-40-> 187 False, 132 True, 114 SemiTrue.
120-80-> 193 False, 132 True, 108 SemiTrue.

Also, I tried to use sobel at hough algorithm but It was a bit slow and vote system was not good at sobel. Also I have implemented hough_lines_max algorithm for canny so It was not good at Sobel. But still I have the results of sobel too.

I have an excel file of my results and my 3 different result folder. Also I did not use annotations and xml files of the dataset.

Example results:
![title](examples/39.png)
![title](examples/133.png)
![title](examples/134.png)
![title](examples/160.png)
![title](examples/173.png)
![title](examples/182.png)
![title](examples/195.png)
![title](examples/243.png)
![title](examples/416.png)


In [28]:


dataset_folder = 'dataset/'

png_files = glob.glob(dataset_folder + '*.png')

if not os.path.exists('output_final_60_40'):
    os.makedirs('output_final_60_40')

for file in png_files:
    template_width = 120
    template_height = 30
    img = Image.open(file)
    img_colour = img
    img = torchvision.transforms.functional.to_tensor(img)
    img = img.numpy()
    img = (img[:3,:,:]*255).astype(np.uint8)
    img_colour = torchvision.transforms.functional.to_tensor(img_colour)
    img_colour = img_colour.numpy()
    img_colour = (img_colour[:3,:,:]*255).astype(np.uint8)

    img_colour = rgb2gray(img_colour).astype(np.uint8)

    img_colour[0] = cv2.GaussianBlur(img_colour[0],(3,3),0)
    cannied = canny(img_colour)
    _hough = hough(cannied)

    horizontal_lines = hough_lines_max(cannied,*_hough,10,0,60)
    vertical_lines = hough_lines_max(cannied,*_hough,10,1,40)

    verticals = draw_lines(cannied, vertical_lines)
    horizontals = draw_lines(cannied, horizontal_lines)

    accumulator = compute_accumulator_array(cannied,img_colour,r_table((15,60),360))
    plate = find_the_plate(accumulator,vertical_lines,horizontal_lines,verticals,horizontals)
    line_1 = plate[0]
    line_2 = plate[1]
    line_3 = [line_1[0],line_1[1],line_2[0],line_2[1],1]
    line_4 = [line_1[2],line_1[3],line_2[2],line_2[3],1]
    plate=[line_1,line_2,line_3,line_4]
    accumulator = draw_lines(accumulator, plate)
    img[0] = np.where(accumulator == 255, 255, img[0])
    img[1] = np.where(accumulator == 255, 0, img[1])
    img[2] = np.where(accumulator == 255, 0, img[2])
    # accumulator = np.where(plate == 255, 100, accumulator)

    file_name = file.split('\\')[-1]
    torchvision.io.write_png(torch.from_numpy(img.astype(np.uint8)),"output_final_60_40/output_"+file_name)
    #torchvision.io.write_png(torch.from_numpy(accumulator.astype(np.uint8)),"test2.png")
